In [19]:
import feedparser
import re
import requests

def _get_arxiv_field(entry, key, default=None):
    # feedparser stores arXiv-specific fields with "arxiv_" prefix
    return getattr(entry, f"arxiv_{key}", default)

def search_arxiv_with_pub_status(
    query,
    categories=None,
    max_results=10,
    sort_by="submittedDate",
    sort_order="descending",
    verify_doi_with_crossref=False
):
    base_url = "http://export.arxiv.org/api/query?"

    search_terms = f"all:{query}"
    if categories:
        cat_query = " OR ".join(f"cat:{c}" for c in categories)
        search_terms = f"({search_terms}) AND ({cat_query})"

    url = (
        f"{base_url}"
        f"search_query={search_terms.replace(' ', '+')}"
        f"&sortBy={sort_by}"
        f"&sortOrder={sort_order}"
        f"&max_results={max_results}"
    )

    feed = feedparser.parse(url)

    papers = []
    for entry in feed.entries:
        doi = _get_arxiv_field(entry, "doi", None)
        journal_ref = _get_arxiv_field(entry, "journal_ref", None)

        pdf_url = next((l.href for l in entry.links if getattr(l, "type", "") == "application/pdf"), None)

        # Heuristic "published" signal:
        # - journal_ref is the strongest arXiv-native signal
        # - DOI is also a strong signal (sometimes present even pre-publication, but usually means published/registered)
        published_signal = bool(journal_ref) or bool(doi)

        crossref_ok = None
        if verify_doi_with_crossref and doi:
            crossref_ok = doi_exists_in_crossref(doi)

        papers.append({
            "title": entry.title.strip(),
            "authors": [a.name for a in entry.authors],
            "published": entry.published,
            "updated": entry.updated,
            "abstract": re.sub(r"\s+", " ", entry.summary).strip(),
            "arxiv_url": entry.id,
            "pdf_url": pdf_url,
            "doi": doi,
            "journal_ref": journal_ref,
            "published_signal": published_signal,
            "crossref_verified": crossref_ok,  # None if not checked
        })

        # Sort: published first, then by updated date (newest first)
    papers.sort(
        key=lambda p: (
            not p["published_signal"],  # False (published) comes before True
            p["updated"]
        ),
        reverse=False
    )

    return papers

def doi_exists_in_crossref(doi: str) -> bool:
    # Crossref "works/{doi}" returns 200 if DOI is registered in Crossref
    url = f"https://api.crossref.org/works/{doi}"
    try:
        r = requests.get(url, timeout=20, headers={"User-Agent": "colab-arxiv-check/1.0 (mailto:you@example.com)"})
        return r.status_code == 200
    except requests.RequestException:
        return False


In [20]:
papers = search_arxiv_with_pub_status(
    query="graph neural networks oversmoothing",
    categories=["cs.LG", "cs.AI"],
    max_results=100,
    verify_doi_with_crossref=True
)

for p in papers[:100]:
    print("\nTitle:", p["title"])
    print("journal_ref:", p["journal_ref"])
    print("doi:", p["doi"])
    print("published_signal:", p["published_signal"])
    print("crossref_verified:", p["crossref_verified"])



Title: Explaining Synergistic Effects in Social Recommendations
journal_ref: None
doi: 10.1145/3774904.3792174
published_signal: True
crossref_verified: False

Title: Conditioned Generative Modeling of Molecular Glues: A Realistic AI Approach for Synthesizable Drug-like Molecules
journal_ref: Biomolecules 2025, 15, 849
doi: 10.3390/biom15060849
published_signal: True
crossref_verified: True

Title: In-Network Collective Operations: Game Changer or Challenge for AI Workloads?
journal_ref: IEEE Computer Jan. 2026
doi: 10.1109/MC.2025.3616048
published_signal: True
crossref_verified: True

Title: Sentipolis: Emotion-Aware Agents for Social Simulations
journal_ref: None
doi: None
published_signal: False
crossref_verified: None

Title: Multimodal Machine Learning for Soft High-k Elastomers under Data Scarcity
journal_ref: None
doi: None
published_signal: False
crossref_verified: None

Title: Leveraging Persistence Image to Enhance Robustness and Performance in Curvilinear Structure Segment